## Pipeline

In [ ]:
# importando a classe Leida
from pyleida import DataLoader, Leida
import numpy as np
np.float_ = np.float64

# instanciando a classe Leida, especificando onde estão os dados

ld = DataLoader('./ArquivosNew/GordonEducation/LEiDA_results')




In [ ]:
ld.occupancies(k=4)

In [ ]:
from pyleida import plotting
import pandas as pd

metric_data = {}
K_min = 2
K_max = 12

for K in range(K_min, K_max):
    df = ld.occupancies(k=K)
    metric_data[f'k_{K}'] = df

stats_all = []

for K in range(K_min, K_max):
    df_stats = ld.stats(k=K, metric='occupancies')
    df_stats['K'] = K
    stats_all.append(df_stats)

stats = pd.concat(stats_all, ignore_index=True)

plotting.plot_pyramid(metric_data,stats,K_min=2,K_max=11)


In [ ]:
from pyleida import plotting
import pandas as pd

metric_data = {}
K_min = 2
K_max = 12

for K in range(K_min, K_max):
    df = ld.dwell_times(k=K)
    metric_data[f'k_{K}'] = df

stats_all = []

for K in range(K_min, K_max):
    df_stats = ld.stats(k=K, metric='dwell_times')
    df_stats['K'] = K
    stats_all.append(df_stats)

stats = pd.concat(stats_all, ignore_index=True)

plotting.plot_pyramid(metric_data,stats,K_min=2,K_max=11)

# K = 4


## load_centroids

In [ ]:
ld.load_centroids(k=4)

## plot_states_on_surf2

## explore_state

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns 
from nilearn.plotting import plot_connectome
import re
import os

def get_states_pvalues(df, k):
    
    if not isinstance(df, pd.DataFrame) or df.empty:
        return {}

    df_k = df[df['k'] == k].copy()

    if df_k.empty:
        return {}

    # extrai número do estado
    df_k['variable'] = df_k['variable'].str.extract(r'PL_state_(\d+)').astype(int)

    # assume que existe coluna 'p_value'
    return dict(zip(df_k['variable'], df_k['p-value']))

def get_color(p, alpha_k, alpha_sumk):
    if p is None:
        return 'gray'
    elif p <= alpha_sumk:
        return 'royalblue'
    elif p <= alpha_k:
        return 'forestgreen'
    else:
        return 'gray'

def centroid2network(centroid):
    N_rois = centroid.size
    network_matrix = np.zeros((N_rois,N_rois))
    for node_idx,node_value in enumerate(centroid):
        if node_value>0:
            for node2_idx,node2_value in enumerate(centroid):
                if node2_value>0:
                    network_matrix[node_idx,node2_idx] = 1
                    network_matrix[node2_idx,node_idx] = 1
    return network_matrix

def plot_states_grid(k, ld, n_cols, save_path, darkstyle=False):

    alpha_k = 0.05 / k
    alpha_sumk = 0.05 / 209   # ou soma real se você tiver Σ(k)

    occ_pvals = get_states_pvalues(ld.significant_states(metric='occupancies'), k)
    dt_pvals = get_states_pvalues(ld.significant_states(metric='dwell_times'), k)
    
    centroids = ld.load_centroids(k=k).values
    n_states = centroids.shape[0]

    fig, axes = plt.subplots(
        4, n_states,
        figsize=(3*n_states, 16),
        constrained_layout=False
    )

    for i in range(n_states):

        centroid = centroids[i, :]
        state = i + 1

        p_occ = occ_pvals.get(state)
        p_dt = dt_pvals.get(state)

        color_occ = get_color(p_occ, alpha_k, alpha_sumk)
        color_dt = get_color(p_dt, alpha_k, alpha_sumk)

        occ = ld.occupancies(k=k)[['condition', f'PL_state_{state}']]
        dt = ld.dwell_times(k=k)[['condition', f'PL_state_{state}']]

        # reaproveita axes por coluna
        axd = {
            'top': axes[0, i],
            'middle': axes[1, i],
            'low': axes[2, i],
            'left': axes[3, i],
        }

        # 🔥 reaproveita código interno (copiando o corpo principal)
        centroid_ = pd.DataFrame(
            {'roi': ld.rois_labels, 'value': centroid}
        ).sort_values('value', ascending=False).reset_index()

        sns.barplot(
            x=centroid_.value,
            y=centroid_.roi,
            palette=['firebrick' if v > 0 else 'royalblue' for v in centroid_.value],
            ax=axd['left']
        )

        axd['left'].axvline(0, color='black')
        axd['left'].set_yticklabels([])
        axd['left'].set_xlabel(None)
        axd['left'].set_ylabel(None)

        # conectoma
        network = centroid2network(centroid)
        sizes = [20 if roi > 0 else 0 for roi in centroid]

        plot_connectome(
            network,
            ld.rois_coordinates_,
            node_color='blue',
            node_size=sizes,
            display_mode='z',
            figure=fig,
            axes=axd['top'],
            annotate=False,
            black_bg=darkstyle,
            alpha=0.1,
            edge_kwargs={'linewidth':1,'color':'firebrick'},
            colorbar=False
        )

        axd['top'].set_title(f'State {state}')

        
        # occupancy
        occ.columns = ['condition','value']
        sns.boxplot(
            x=occ.condition,
            y=occ.value,
            ax=axd['middle'],
            color=color_occ
        )

        # dwell time
        dt.columns = ['condition','value']
        sns.boxplot(
            x=dt.condition,
            y=dt.value,
            ax=axd['low'],
            color=color_dt
        )

        axd['middle'].set_xlabel('')
        axd['middle'].set_ylabel('')
        axd['middle'].set_title('Occupancies', fontsize=12, pad=10)
        #if is_sig_occ:
        #    axd['middle'].set_facecolor('#fff3cd')
              # amarelo claro
        axd['low'].set_xlabel('')
        axd['low'].set_ylabel('')
        axd['low'].set_title('Dwell Times', fontsize=12, pad=10)
        #if is_sig_dt:
        #    axd['low'].set_facecolor('#fff3cd')  # azul claro
    
    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
plot_states_grid(
    k=10,
    ld=ld,
    n_cols=10,
    save_path="k10_all_states.png"
)

In [ ]:
ld.explore_state(
    k=4,
    state=1,
    darkstyle=False
)
ld.explore_state(
    k=4,
    state=2,
    darkstyle=False
)
ld.explore_state(
    k=4,
    state=3,
    darkstyle=False
)
ld.explore_state(
    k=4,
    state=4,
    darkstyle=False
)

## plot_state_nodes

### State 1

In [ ]:
ld.plot_states_nodes(k=4,state=1,show_labels=False)

### State 2

In [ ]:
ld.plot_states_nodes(k=4,state=2,show_labels=False)

### State 3

In [ ]:
ld.plot_states_nodes(k=4,state=3,show_labels=False)

### State 4

In [ ]:
ld.plot_states_nodes(k=4,state=4,show_labels=False)

# K = 10

## load_centroids

In [ ]:
ld.load_centroids(k=10)

## plot_states_on_surf2

### State 1

In [ ]:
ld.plot_states_on_surf2(k=10, state=1, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 2

In [ ]:
ld.plot_states_on_surf2(k=10, state=2, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 3

In [ ]:
ld.plot_states_on_surf2(k=10, state=3, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 4

In [ ]:
ld.plot_states_on_surf2(k=10, state=4, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 5

In [ ]:
ld.plot_states_on_surf2(k=10, state=5, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 6

In [ ]:
ld.plot_states_on_surf2(k=10, state=6, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 7

In [ ]:
ld.plot_states_on_surf2(k=10, state=7, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 8

In [ ]:
ld.plot_states_on_surf2(k=10, state=8, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 9

In [ ]:
ld.plot_states_on_surf2(k=10, state=9, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

### State 10

In [ ]:
ld.plot_states_on_surf2(k=10, state=10, parcellation='./Atlas/gordon333MNI.nii.gz', surface='white', hemi='both')

## explore_state

In [ ]:
ld.explore_state(
    k=10,
    state=1,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=2,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=3,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=4,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=5,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=6,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=7,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=8,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=9,
    darkstyle=False
)
ld.explore_state(
    k=10,
    state=10,
    darkstyle=False
)


## plot_state_nodes

### State 1

In [ ]:
ld.plot_states_nodes(k=10,state=1,show_labels=False)

### State 2

In [ ]:
ld.plot_states_nodes(k=10,state=2,show_labels=False)

### State 3

In [ ]:
ld.plot_states_nodes(k=10,state=3,show_labels=False)

### State 4

In [ ]:
ld.plot_states_nodes(k=10,state=4,show_labels=False)

### State 5

In [ ]:
ld.plot_states_nodes(k=10,state=5,show_labels=False)

### State 6

In [ ]:
ld.plot_states_nodes(k=10,state=6,show_labels=False)

### State 7

In [ ]:
ld.plot_states_nodes(k=10,state=7,show_labels=False)

### State 8

In [ ]:
ld.plot_states_nodes(k=10,state=8,show_labels=False)

### State 9

In [ ]:
ld.plot_states_nodes(k=10,state=9,show_labels=False)

### State 10

In [ ]:
ld.plot_states_nodes(k=10,state=10,show_labels=False)